# Session 2: the mathematics of an orbit
**A. Real-world problem.** An operator needs an orbit that dips to 500 km above the ground
and climbs to 35786 km. How long is one lap, how fast is the satellite at each end, and where
is it one hour after perigee? You need four tools: circular speed, the vis-viva equation,
Kepler's third law, and Kepler's equation. Then you check the physics by simulation, using two
numerical methods and measuring how much energy each one wrongly gains or loses.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
MU = 3.986004418e14      # m^3/s^2
R_EARTH = 6.371e6        # m

## Step 1: circular orbits and Kepler's third law
v = sqrt(mu / r) and T = 2 pi sqrt(a^3 / mu), where a is the semi-major axis (half the long
axis of the ellipse; for a circle a = r).

In [ ]:
def circular_speed(r):
    return np.sqrt(MU / r)


def period(a):
    """Orbital period (s) from the semi-major axis a (m): Kepler's third law."""
    return 2 * np.pi * np.sqrt(a**3 / MU)

## Step 2: the ellipse and the vis-viva equation
Closest point = perigee radius rp; farthest = apogee radius ra (both measured from the centre
of the Earth, so altitude + 6371 km). Then a = (rp + ra) / 2 and e = (ra - rp) / (ra + rp).
The speed anywhere is given by vis-viva: v^2 = mu (2/r - 1/a).

In [ ]:
def ellipse_from_peri_apo(rp, ra):
    """Return semi-major axis a (m) and eccentricity e from perigee and apogee radii (m)."""
    return (rp + ra) / 2, (ra - rp) / (ra + rp)


def vis_viva(r, a):
    """Speed (m/s) at distance r (m) on an orbit of semi-major axis a (m)."""
    return np.sqrt(MU * (2 / r - 1 / a))


def specific_energy(r, v):
    """Energy per kilogram (J/kg): kinetic minus gravitational. Equals -mu / (2 a) for a Kepler orbit."""
    return 0.5 * v**2 - MU / r


def angular_momentum(rvec, vvec):
    """Specific angular momentum (m^2/s): x*vy - y*vx. Conserved because gravity points at the centre."""
    return rvec[0] * vvec[1] - rvec[1] * vvec[0]


def semi_major_axis_for_period(T):
    """Semi-major axis (m) for a period T (s): Kepler's third law solved for a."""
    return (MU * (T / (2 * np.pi)) ** 2) ** (1 / 3)


def apogee_altitude_for_period(T, alt_perigee_km):
    """Apogee altitude (km) of the orbit with period T (s) and the given perigee altitude (km)."""
    return (2 * semi_major_axis_for_period(T) - (R_EARTH + alt_perigee_km * 1e3) - R_EARTH) / 1e3


def design_orbit(alt_perigee_km, alt_apogee_km):
    """Everything about an orbit with given perigee and apogee altitudes."""
    rp, ra = R_EARTH + alt_perigee_km * 1e3, R_EARTH + alt_apogee_km * 1e3
    a, e = ellipse_from_peri_apo(rp, ra)
    return {"a": a, "e": e, "T": period(a), "vp": vis_viva(rp, a), "va": vis_viva(ra, a), "rp": rp, "ra": ra}

## Step 3: Kepler's equation, where is the satellite at time t?
Mean anomaly M = 2 pi t / T grows steadily. The eccentric anomaly E is the geometric angle
we need; they are linked by M = E - e sin(E). There is no formula for E, so we solve it by
Newton's method: E <- E - (E - e sin E - M) / (1 - e cos E).

In [ ]:
def solve_kepler(M, e, tol=1e-12, max_iter=50):
    """Solve M = E - e sin E for E (radians). Returns (E, iterations, residual)."""
    E = M if e < 0.8 else np.pi
    for k in range(1, max_iter + 1):
        f = E - e * np.sin(E) - M
        E = E - f / (1 - e * np.cos(E))
        if abs(E - e * np.sin(E) - M) < tol:
            return E, k, abs(E - e * np.sin(E) - M)
    return E, max_iter, abs(E - e * np.sin(E) - M)


def radius_from_E(E, a, e):
    return a * (1 - e * np.cos(E))

## Step 4: integrate the two-body equations and measure the drift
Acceleration is -mu r / |r|^3. RK4 is a general-purpose method (accurate each step, but its
energy slowly drifts). Leapfrog (velocity Verlet) is *symplectic*: its energy wobbles but does
not drift. We measure both over many laps.

In [ ]:
def accel(r):
    return -MU * r / np.linalg.norm(r) ** 3


def rk4_step(r, v, dt):
    k1r, k1v = v, accel(r)
    k2r, k2v = v + 0.5 * dt * k1v, accel(r + 0.5 * dt * k1r)
    k3r, k3v = v + 0.5 * dt * k2v, accel(r + 0.5 * dt * k2r)
    k4r, k4v = v + dt * k3v, accel(r + dt * k3r)
    return r + dt / 6 * (k1r + 2 * k2r + 2 * k3r + k4r), v + dt / 6 * (k1v + 2 * k2v + 2 * k3v + k4v)


def leapfrog_step(r, v, dt):
    v_half = v + 0.5 * dt * accel(r)
    r_new = r + dt * v_half
    return r_new, v_half + 0.5 * dt * accel(r_new)


def propagate(step, r0, v0, dt, n):
    """Run n steps. Returns arrays of positions (n+1, 2), energy (n+1) and angular momentum (n+1)."""
    r, v = np.array(r0, float), np.array(v0, float)
    pos, en, hh = [r.copy()], [specific_energy(np.linalg.norm(r), np.linalg.norm(v))], [angular_momentum(r, v)]
    for _ in range(n):
        r, v = step(r, v, dt)
        pos.append(r.copy())
        en.append(specific_energy(np.linalg.norm(r), np.linalg.norm(v)))
        hh.append(angular_momentum(r, v))
    return np.array(pos), np.array(en), np.array(hh)


def perigee_start(rp, ra):
    """Start state (position, velocity) at perigee on the x axis, counter-clockwise."""
    a, _ = ellipse_from_peri_apo(rp, ra)
    return np.array([rp, 0.0]), np.array([0.0, vis_viva(rp, a)])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    o = design_orbit(500, 35786)
    print(f"design orbit 500 x 35786 km: a = {o['a'] / 1e3:.0f} km, e = {o['e']:.4f}, T = {o['T'] / 3600:.3f} h")
    print(f"  perigee speed {o['vp'] / 1e3:.3f} km/s, apogee speed {o['va'] / 1e3:.3f} km/s, ratio {o['vp'] / o['va']:.3f} = ra/rp {o['ra'] / o['rp']:.3f}")
    print(f"  energy -mu/2a = {-MU / (2 * o['a']):.1f} J/kg")
    print(f"12 hour orbit with perigee 500 km: a = {semi_major_axis_for_period(43200.0) / 1e3:.0f} km, apogee altitude {apogee_altitude_for_period(43200.0, 500):.0f} km")
    E, k, res = solve_kepler(2 * np.pi * 3600 / o["T"], o["e"])
    print(f"  one hour after perigee: M = {2 * np.pi * 3600 / o['T']:.4f} rad, E = {E:.5f} rad, {k} iterations, residual {res:.1e}, r = {radius_from_E(E, o['a'], o['e']) / 1e3:.0f} km")
    for e in (0.0, 0.3, 0.7, 0.95):
        its = max(solve_kepler(M, e)[1] for M in np.linspace(0.01, 6.27, 200))
        print(f"  e = {e}: at most {its} Newton iterations")

    rp, ra = 7.0e6, 2.1e7
    r0, v0 = perigee_start(rp, ra)
    T = period((rp + ra) / 2)
    dt, laps = 60.0, 50
    n = int(round(T * laps / dt))
    res = {}
    for name, step in (("RK4", rk4_step), ("leapfrog", leapfrog_step)):
        pos, en, hh = propagate(step, r0, v0, dt, n)
        res[name] = (pos, en, hh)
        print(f"{name:9s} dt={dt:.0f}s {laps} laps: energy change {(en[-1] - en[0]) / abs(en[0]):.2e}, worst {np.max(np.abs(en - en[0])) / abs(en[0]):.2e}, h change {(hh[-1] - hh[0]) / hh[0]:.2e}")

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    th = np.linspace(0, 2 * np.pi, 300)
    axs[0].fill(R_EARTH / 1e3 * np.cos(th), R_EARTH / 1e3 * np.sin(th), color="#9ecae1")
    pos = res["RK4"][0][: int(T / dt) + 1] / 1e3
    axs[0].plot(pos[:, 0], pos[:, 1], "k-", lw=1)
    axs[0].plot([rp / 1e3], [0], "ro"); axs[0].text(rp / 1e3 + 500, 800, "perigee")
    axs[0].plot([-ra / 1e3], [0], "go"); axs[0].text(-ra / 1e3, 1500, "apogee")
    axs[0].set_aspect("equal"); axs[0].set_xlabel("x (km)"); axs[0].set_ylabel("y (km)")
    axs[0].set_title("One lap of a 7000 x 21000 km ellipse"); axs[0].grid(alpha=0.3)
    t_h = np.arange(n + 1) * dt / 3600
    for name, col in (("RK4", "#d62728"), ("leapfrog", "#1f77b4")):
        axs[1].plot(t_h, (res[name][1] - res[name][1][0]) / abs(res[name][1][0]), color=col, label=name)
    axs[1].set_xlabel("time (hours)"); axs[1].set_ylabel("relative energy change (no unit)")
    axs[1].set_title("RK4 drifts steadily, leapfrog stays bounded"); axs[1].legend(); axs[1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(OUT / "orbit_and_drift.png", dpi=120)

    M = np.linspace(0, 2 * np.pi, 200)
    fig2, ax2 = plt.subplots(figsize=(5.5, 4))
    for e in (0.0, 0.3, 0.7, 0.9):
        ax2.plot(M, [solve_kepler(m, e)[0] for m in M], label=f"e = {e}")
    ax2.set_xlabel("mean anomaly M (radians)"); ax2.set_ylabel("eccentric anomaly E (radians)")
    ax2.set_title("Solving Kepler's equation"); ax2.legend(); ax2.grid(alpha=0.3)
    fig2.tight_layout(); fig2.savefig(OUT / "kepler_equation.png", dpi=120)

## Coding tasks
1. **Easy.** Use `design_orbit(300, 20000)` to print its period and the two speeds. Predict which speed is bigger before you run it.
2. **Stretch.** Reduce the step in `propagate` from 60 s to 30 s and compare the energy error of both methods. Which one improves more, and by what factor?
3. **Explain.** Explain in words why Newton's method for Kepler's equation needs more iterations for e = 0.95 than for e = 0.3.

## Assessment
- **A. Conceptual.** Why must a satellite move faster at perigee than at apogee? Answer using both energy and angular momentum.
- **B. Numerical.** Use Kepler's third law to find the period of an orbit with semi-major axis 24514 km. Check units and compare with the geostationary period.
- **C. Graph and simulation interpretation.** From the energy figure, which integrator would you trust for a simulation of 1000 laps, and why? Which is more accurate for 10 laps?
- **D. Design and optimisation challenge.** Design: you need a 12 hour orbit that passes 500 km above the ground at perigee. What apogee altitude do you need, and what is the semi-major axis? How would you choose between this orbit and a circular one with the same period?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    r = L.R_EARTH + 400e3
    checks.append(("400 km circular orbit: 7.67 km/s and 92.4 min",
                   abs(L.circular_speed(r) / 1e3 - 7.67) < 0.005 and abs(L.period(r) / 60 - 92.4) < 0.05))
    checks.append((f"geostationary radius 42164 km gives a period of 86164 s (got {L.period(42164e3):.1f})", abs(L.period(42164e3) - 86164) < 1.0))
    checks.append(("the geostationary altitude is 35786 km above the equator (42164 - 6378)", abs((42164 - 6378) - 35786) < 1e-9))

    o = L.design_orbit(500, 35786)
    checks.append((f"worked example: a = 24514 km and e = 0.7197 (got {o['a'] / 1e3:.0f}, {o['e']:.4f})",
                   abs(o["a"] / 1e3 - 24514) < 0.5 and abs(o["e"] - 0.7197) < 5e-5))
    checks.append((f"worked example: period 10.610 h (got {o['T'] / 3600:.3f})", abs(o["T"] / 3600 - 10.610) < 5e-4))
    checks.append((f"worked example: perigee speed 9.988 km/s, apogee speed 1.628 km/s (got {o['vp'] / 1e3:.3f}, {o['va'] / 1e3:.3f})",
                   abs(o["vp"] / 1e3 - 9.988) < 5e-4 and abs(o["va"] / 1e3 - 1.628) < 5e-4))
    checks.append(("angular momentum: rp*vp = ra*va", abs(o["rp"] * o["vp"] / (o["ra"] * o["va"]) - 1) < 1e-12))
    checks.append(("speed ratio equals distance ratio, 6.135", abs(o["vp"] / o["va"] - 6.135) < 5e-4))
    checks.append(("vis-viva with a = r is the circular speed", abs(L.vis_viva(r, r) - L.circular_speed(r)) < 1e-9))
    en_p = L.specific_energy(o["rp"], o["vp"])
    en_a = L.specific_energy(o["ra"], o["va"])
    checks.append(("energy -mu/(2a) = -8.13 MJ/kg at both ends", abs(en_p - en_a) < 1e-3 and abs(en_p + L.MU / (2 * o["a"])) < 1e-3 and abs(en_p / 1e6 + 8.13) < 0.005))
    a12 = L.semi_major_axis_for_period(43200.0)
    ap12 = L.apogee_altitude_for_period(43200.0, 500)
    o12 = L.design_orbit(500, ap12)
    checks.append((f"design: a 12 hour orbit has a = 26610 km, and with perigee 500 km its apogee altitude is 39978 km (got {a12 / 1e3:.0f}, {ap12:.0f})",
                   abs(a12 / 1e3 - 26610) < 0.5 and abs(ap12 - 39978) < 0.5 and abs(o12["T"] - 43200.0) < 1e-6))
    a, e = L.ellipse_from_peri_apo(7e6, 7e6)
    checks.append(("a circle has e = 0", e == 0 and a == 7e6))

    # Kepler's equation
    worst_res, worst_it = 0.0, 0
    for ecc in (0.0, 0.3, 0.7, 0.95):
        for M in np.linspace(0.0, 2 * np.pi, 120):
            E, it, res = L.solve_kepler(M, ecc)
            worst_res = max(worst_res, abs(E - ecc * np.sin(E) - M))
            worst_it = max(worst_it, it)
    checks.append((f"Kepler's equation residual below 1e-12 for e up to 0.95 (worst {worst_res:.1e})", worst_res < 1e-12))
    checks.append((f"Newton needs at most 8 iterations (worst {worst_it})", worst_it <= 8))
    M1 = 2 * np.pi * 3600 / o["T"]
    E1, it1, _ = L.solve_kepler(M1, o["e"])
    checks.append((f"one hour after perigee: E = 1.2821 rad, r = 19491 km in 5 iterations (got {E1:.4f}, {L.radius_from_E(E1, o['a'], o['e']) / 1e3:.0f}, {it1})",
                   abs(E1 - 1.2821) < 5e-4 and abs(L.radius_from_E(E1, o["a"], o["e"]) / 1e3 - 19491) < 2 and it1 == 5))
    checks.append(("edge: e = 0 gives E = M, M = 0 gives E = 0, M = pi gives E = pi",
                   abs(L.solve_kepler(1.234, 0.0)[0] - 1.234) < 1e-12 and abs(L.solve_kepler(0.0, 0.9)[0]) < 1e-12
                   and abs(L.solve_kepler(np.pi, 0.9)[0] - np.pi) < 1e-9))

    # integrators on a 7000 x 21000 km orbit
    rp, ra = 7.0e6, 2.1e7
    r0, v0 = L.perigee_start(rp, ra)
    T = L.period((rp + ra) / 2)
    dt = 60.0
    k = int(round(T / dt))
    pos, en, hh = L.propagate(L.rk4_step, r0, v0, T / k, k)
    checks.append((f"RK4 returns to the start point after one period (within 2 km, got {np.linalg.norm(pos[-1] - r0) / 1e3:.2f})", np.linalg.norm(pos[-1] - r0) < 2e3))
    d = {}
    for name, step in (("rk4", L.rk4_step), ("lf", L.leapfrog_step)):
        p10, e10, h10 = L.propagate(step, r0, v0, dt, 10 * k)
        p40, e40, h40 = L.propagate(step, r0, v0, dt, 40 * k)
        rel10 = abs(e10[-1] - e10[0]) / abs(e10[0])
        rel40 = abs(e40[-1] - e40[0]) / abs(e40[0])
        d[name] = (rel10, rel40, np.max(np.abs(e40 - e40[0])) / abs(e40[0]), abs(h40[-1] - h40[0]) / h40[0])
    checks.append((f"RK4 energy drift grows with laps (10 laps {d['rk4'][0]:.1e}, 40 laps {d['rk4'][1]:.1e})", d["rk4"][1] > 2.5 * d["rk4"][0]))
    checks.append((f"leapfrog energy error stays bounded: worst over 40 laps below 2e-3 (got {d['lf'][2]:.1e})", d["lf"][2] < 2e-3))
    checks.append(("leapfrog conserves angular momentum to 1e-12", d["lf"][3] < 1e-12))
    checks.append((f"RK4 energy change: 1.9e-6 after 10 laps, 7.8e-6 after 40 (got {d['rk4'][0]:.1e}, {d['rk4'][1]:.1e})",
                   abs(d["rk4"][0] - 1.9e-6) < 0.1e-6 and abs(d["rk4"][1] - 7.8e-6) < 0.2e-6))
    checks.append(("with this step, RK4 is more accurate than leapfrog after 40 laps", d["rk4"][1] < d["lf"][1]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
